# Iftixor — Google Colab orqali GPU bilan o'qitish

Bu notebook Iftixor modelini bepul Colab GPU'sida tez o'qitish uchun.

**Avval yoqing:** Yuqoridagi menyudan `Runtime -> Change runtime type -> T4 GPU` ni tanlang, so'ng har bir katakchani tepadan pastga qarab ishga tushiring (Shift+Enter).

In [5]:
import torch
print("GPU mavjud:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU nomi:", torch.cuda.get_device_name(0))
else:
    print("GPU topilmadi — Runtime -> Change runtime type -> T4 GPU ni tanlab, qayta ishga tushiring.")

GPU mavjud: True
GPU nomi: Tesla T4


## 1. Repozitoriyani yuklab olish

In [7]:
!git clone https://github.com/Iftix0r/Iftixor.git
%cd Iftixor
!pip install -q python-dotenv

Cloning into 'Iftixor'...
remote: Enumerating objects: 64, done.
remote: Counting objects: 100% (64/64), done.
remote: Compressing objects: 100% (44/44), done.
remote: Total 64 (delta 26), reused 52 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (64/64), 23.13 KiB | 3.85 MiB/s, done.
Resolving deltas: 100% (26/26), done.
/content/Iftixor/Iftixor


## 2. (Ixtiyoriy) Serveringizdan `conversations.txt` yoki boshqa qo'shimcha ma'lumot yuklash

Agar serveringizdagi `data/conversations.txt` yoki boshqa matn faylini shu yerda ham qo'shib o'qitmoqchi bo'lsangiz, uni avval o'z kompyuteringizga yuklab oling (`scp` bilan), so'ng shu katakchani ishga tushirib, faylni tanlang. Agar kerak bo'lmasa, bu katakchani o'tkazib yuboring.

In [8]:
from google.colab import files
import shutil

uploaded = files.upload()
for fname in uploaded.keys():
    shutil.move(fname, f"data/{fname}")
    print(f"data/{fname} ga joylandi")

## 3. O'qitish (GPU bilan)

GPU bo'lgani uchun `--batch-size`ni kattaroq, `--steps`ni ham ko'proq qilish mumkin — CPU'dagi soatlab vaqt o'rniga bu bir necha daqiqada tugaydi.

Agar 2-qadamda qo'shimcha fayl yuklagan bo'lsangiz, uni pastdagi `--data` ro'yxatiga qo'shing (masalan `data/conversations.txt`).

In [9]:
!python -m iftixor.train \
  --data data/corpus.txt data/synthetic.txt \
  --steps 5000 \
  --batch-size 64 \
  --device auto \
  --out checkpoints/iftixor.pt

Ogohlantirish: data/synthetic.txt topilmadi, o'tkazib yuborildi.
Vocab hajmi: 56 | Parametrlar soni: 824,064 | Qurilma: cuda
step 200/5000 | train loss 1.7481 | val loss 2.6917 | 6.7s
step 400/5000 | train loss 0.4850 | val loss 3.8945 | 12.6s
step 600/5000 | train loss 0.1890 | val loss 4.1487 | 18.5s
step 800/5000 | train loss 0.1412 | val loss 4.6781 | 24.5s
step 1000/5000 | train loss 0.1077 | val loss 4.6773 | 30.5s
step 1200/5000 | train loss 0.0997 | val loss 4.7624 | 36.5s
step 1400/5000 | train loss 0.0810 | val loss 4.8713 | 42.6s
step 1600/5000 | train loss 0.0743 | val loss 5.1192 | 48.8s
step 1800/5000 | train loss 0.0727 | val loss 4.9241 | 55.0s
step 2000/5000 | train loss 0.0682 | val loss 5.0910 | 61.2s
step 2200/5000 | train loss 0.0628 | val loss 5.2601 | 67.5s
step 2400/5000 | train loss 0.0564 | val loss 5.2505 | 73.9s
step 2600/5000 | train loss 0.0621 | val loss 5.4560 | 80.4s
step 2800/5000 | train loss 0.0539 | val loss 5.3520 | 86.9s
step 3000/5000 | train los

## 4. Terminalda tez sinash (ixtiyoriy)

In [11]:
from iftixor.generate import load_model, generate_text

model, tokenizer = load_model("checkpoints/iftixor.pt")
prompt = "Foydalanuvchi: Salom!\nIftixor:"
print(generate_text(model, tokenizer, prompt, max_new_tokens=100))

Foydalanuvchi: Salom!
Iftixor: Salom! Men Iftixor, sizga qanday yordam bera olaman?

Foydalanuvchi: Ismingiz nima?
Iftixor: Mening


## 5. Tayyor checkpointni yuklab olish

Bu faylni kompyuteringizga saqlab, so'ng `scp` yoki SFTP orqali serveringizdagi `checkpoints/iftixor.pt` fayli o'rniga joylashtiring, keyin serverda botni qayta ishga tushiring (`systemctl restart iftixor`) yoki Telegram'da `/reload` buyrug'ini yuboring.

In [12]:
from google.colab import files
files.download("checkpoints/iftixor.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>